# Training YOLO to automate labeling process

### Environment Setup

In [ ]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import torch
import albumentations as A
import numpy as np

# TODO: Update this path to your dataset configuration file
data_path = "./data/processed/data.yaml"

### Training Model

In [ ]:
# 加载 YOLO 模型
model = YOLO('yolo11l.pt')  # 使用预训练的 YOLO11x 模型 

# 开始训练
results = model.train(
    data=data_path,  # 数据集配置文件路径
    # imgsz=640,                 # 图片大小
    epochs=500,                 # 训练轮数
    batch=10,                  # 批量大小
    name='mahjong-yolom',       # 训练结果保存的项目名称
    pretrained=True,            # 是否使用预训练权重
    device='cuda',
    lr0=0.01,              # 初始学习率，视情况调整
    patience=20,
    half=True,
    cache='ram',
    workers=8,
)


### Test prediction

In [ ]:
# 调整边界框宽度和字体大小
def draw_boxes_with_custom_style(image, boxes, labels, line_thickness=0.1, font_scale=0.1, font_thickness=0.1):
    """
    在图片上绘制边界框，支持自定义边框宽度和字体大小。

    :param image: 原始图片
    :param boxes: 边界框信息
    :param labels: 类别标签
    :param line_thickness: 边界框线条宽度
    :param font_scale: 字体大小
    :param font_thickness: 字体粗细
    :return: 绘制边界框后的图片
    """
    for box in boxes:
        cls = int(box.cls)  # 类别索引
        conf = float(box.conf)  # 置信度
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # 边界框坐标

        # 绘制边界框
        cv2.rectangle(image, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=line_thickness)

        # 绘制标签
        label = f"{labels[cls]}: {conf:.2f}"
        label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
        label_x = x1
        label_y = y1 - 10 if y1 - 10 > 10 else y1 + 10
        cv2.rectangle(image, (label_x, label_y - label_size[1]), (label_x + label_size[0], label_y), (0, 255, 0), -1)
        cv2.putText(image, label, (label_x, label_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), font_thickness)
    
    return image

# TODO: Update this path to your test image location
image_path = './data/processed/images/test/test_image.png'

# 进行预测
results = model.predict(source=image_path, conf=0.25)

# 获取检测结果
boxes = results[0].boxes  # 检测到的边界框
labels = results[0].names  # 类别标签
image = cv2.imread(image_path)

# 绘制边界框，调整线宽和字体大小
image_with_custom_boxes = draw_boxes_with_custom_style(
    image.copy(), boxes, labels, line_thickness=1, font_scale=0.8, font_thickness=1
)

# 保存结果到文件（可选）
output_path = "detection_results.png"
cv2.imwrite(output_path, image_with_custom_boxes)

# 显示检测结果图片
plt.figure(figsize=(12, 12))
plt.imshow(cv2.cvtColor(image_with_custom_boxes, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Detection Results with Custom Style")
plt.show()